In [ ]:
# !unzip /content/playground-series-s6e2.zip

# 📌Install & Imports

In [ ]:
# !pip install --quiet scikit-learn==1.7.2 catboost==1.2.8

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay
from sklearn.ensemble import HistGradientBoostingClassifier
import optuna


from catboost import CatBoostClassifier
from copy import deepcopy
import warnings
warnings.filterwarnings("ignore")

SEED = 42

# 📂 Load Data

In [ ]:
TARGET = "Heart Disease"

train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv', index_col='id')
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv', index_col='id')
orig = pd.read_csv('/kaggle/input/datasets/neurocipher/heartdisease/Heart_Disease_Prediction.csv')

# Merge external dataset
train = pd.concat([train, orig[train.columns]], axis=0, ignore_index=True)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

In [ ]:
train.head()

In [ ]:
y = (train[TARGET] == "Presence").astype(int)
X = train.drop(columns=[TARGET]).copy()
X_test = test.copy()

# ⚙️ Preprocessing 

In [ ]:
num_features = X.select_dtypes(exclude=['object', 'bool']).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'bool']).columns.tolist()

# Combine for global feature engineering
combine = pd.concat([X, X_test])

# Rank categorical from numeric
rank_cols = []
for col in num_features:
    new_col = f"{col}_cat"
    combine[new_col] = combine[col].rank(method="dense").astype(int)
    rank_cols.append(new_col)

# Interaction
combine["ST_Slope"] = combine["ST depression"] * combine["Slope of ST"]

# Cast categorical globally
combine[cat_features] = combine[cat_features].astype("category")
combine[rank_cols] = combine[rank_cols].astype("category")

# Split back
X = combine.iloc[:len(X)].copy()
X_test = combine.iloc[len(X):].copy()

cat_features = X.select_dtypes(include=["category"]).columns.tolist()
num_features = X.select_dtypes(exclude=["category"]).columns.tolist()

print("Total Features:", len(X.columns))

In [ ]:
def fit_frequency_encoding(df, cols):
    freq_dict = {}
    for col in cols:
        freq_dict[col] = df[col].value_counts(normalize=True).to_dict()
    return freq_dict

def apply_frequency_encoding(df, freq_dict):
    df = df.copy()
    for col, mapping in freq_dict.items():
        df[f"{col}_fe"] = df[col].map(mapping).astype(float).fillna(0)
    return df

all_cats = cat_features.copy()

freq_dict = fit_frequency_encoding(X, all_cats)

X = apply_frequency_encoding(X, freq_dict)
X_test = apply_frequency_encoding(X_test, freq_dict)

print("After FE:", X.shape)

In [ ]:
def objective_hgb(trial):

    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 255),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 20, 200),
        "l2_regularization": trial.suggest_float("l2_regularization", 0.0, 5.0),
        "max_iter": 5000,
        "early_stopping": True,
        "validation_fraction":0.1,
        "n_iter_no_change":100,
        "random_state": SEED
    }

    skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=SEED)
    scores = []

    for train_idx, valid_idx in skf.split(X, y):

        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
        y_train, y_val = y.iloc[train_idx], y.iloc[valid_idx]

        # Target Encoding
        te = TargetEncoder(random_state=42, cv=5)
        X_train[cat_features] = te.fit_transform(X_train[cat_features], y_train)
        X_val[cat_features] = te.transform(X_val[cat_features])

        model = HistGradientBoostingClassifier(**params)
        model.fit(X_train, y_train)

        preds = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, preds))

    return np.mean(scores)

In [ ]:
study_hgb = optuna.create_study(direction="maximize")
study_hgb.optimize(objective_hgb, n_trials=10)

print("Best HGB AUC:", study_hgb.best_value)
print("Best Params:", study_hgb.best_params)

# 🚀 Model Building

In [ ]:
best_hgb_params = study_hgb.best_params
best_hgb_params.update({
    "max_iter": 10000,
    "early_stopping": True,
    "validation_fraction":0.1,
    "n_iter_no_change":100,
    "random_state": SEED
})

skf = StratifiedKFold(n_splits=7, shuffle=True, random_state=SEED)

oof_hgb = np.zeros(len(X))
test_hgb = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):

    print(f"\n===== HGB Fold {fold+1} =====")

    X_train = X.iloc[train_idx].copy()
    y_train = y.iloc[train_idx]

    X_val = X.iloc[valid_idx].copy()
    y_val = y.iloc[valid_idx]

    X_t = X_test.copy()

    te = TargetEncoder(random_state=42, cv=5)
    X_train[cat_features] = te.fit_transform(X_train[cat_features], y_train)
    X_val[cat_features] = te.transform(X_val[cat_features])
    X_t[cat_features] = te.transform(X_t[cat_features])

    model = HistGradientBoostingClassifier(**best_hgb_params)
    model.fit(X_train, y_train)

    oof_hgb[valid_idx] = model.predict_proba(X_val)[:, 1]
    test_hgb += model.predict_proba(X_t)[:, 1] / skf.n_splits

    print("Fold AUC:", roc_auc_score(y_val, oof_hgb[valid_idx]))

print("🔥 HGB OOF AUC:", roc_auc_score(y, oof_hgb))

In [ ]:
pd.DataFrame({"hgb_oof": oof_hgb}).to_csv("hgb_oof.csv", index=False)
pd.DataFrame({"hgb_test": test_hgb}).to_csv("hgb_test.csv", index=False)

In [ ]:
# from sklearn.ensemble import ExtraTreesClassifier

# def objective_et(trial):

#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 300, 1000),
#         "max_depth": trial.suggest_int("max_depth", 5, 12),
#         "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
#         "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
#         "max_features": trial.suggest_float("max_features", 0.3, 1.0),
#         "n_jobs": -1,
#         "random_state": SEED
#     }

#     skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=SEED)
#     scores = []

#     for train_idx, valid_idx in skf.split(X, y):

#         X_train, X_val = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
#         y_train, y_val = y.iloc[train_idx], y.iloc[valid_idx]

#         te = TargetEncoder(random_state=42, cv=5)
#         X_train[cat_features] = te.fit_transform(X_train[cat_features], y_train)
#         X_val[cat_features] = te.transform(X_val[cat_features])

#         model = ExtraTreesClassifier(**params)
#         model.fit(X_train, y_train)

#         preds = model.predict_proba(X_val)[:, 1]
#         scores.append(roc_auc_score(y_val, preds))

#     return np.mean(scores)

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

In [ ]:
def objective_ada(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.0),
        "random_state": SEED
    }

    skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=SEED)
    scores = []

    for train_idx, valid_idx in skf.split(X, y):

        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
        y_train, y_val = y.iloc[train_idx], y.iloc[valid_idx]

        te = TargetEncoder(random_state=42, cv=5)
        X_train[cat_features] = te.fit_transform(X_train[cat_features], y_train)
        X_val[cat_features] = te.transform(X_val[cat_features])

        model = AdaBoostClassifier(**params)
        model.fit(X_train, y_train)

        preds = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, preds))

    return np.mean(scores)

In [ ]:
study_ada = optuna.create_study(direction="maximize")
study_ada.optimize(objective_ada, n_trials=10)

print("Best Ada AUC:", study_ada.best_value)
print("Best Params:", study_ada.best_params)

In [ ]:
best_ada_params = study_ada.best_params
best_ada_params.update({"random_state": SEED})

skf = StratifiedKFold(n_splits=7, shuffle=True, random_state=SEED)

oof_ada = np.zeros(len(X))
test_ada = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):

    print(f"\n===== ADA Fold {fold+1} =====")

    X_train = X.iloc[train_idx].copy()
    y_train = y.iloc[train_idx]

    X_val = X.iloc[valid_idx].copy()
    y_val = y.iloc[valid_idx]

    X_t = X_test.copy()

    te = TargetEncoder(random_state=42, cv=5)
    X_train[cat_features] = te.fit_transform(X_train[cat_features], y_train)
    X_val[cat_features] = te.transform(X_val[cat_features])
    X_t[cat_features] = te.transform(X_t[cat_features])

    model = AdaBoostClassifier(**best_ada_params)
    model.fit(X_train, y_train)

    oof_ada[valid_idx] = model.predict_proba(X_val)[:, 1]
    test_ada += model.predict_proba(X_t)[:, 1] / skf.n_splits

    print("Fold AUC:", roc_auc_score(y_val, oof_ada[valid_idx]))

print("🔥 ADA OOF AUC:", roc_auc_score(y, oof_ada))


In [ ]:
pd.DataFrame({"ada_oof": oof_ada}).to_csv("ada_oof.csv", index=False)
pd.DataFrame({"ada_test": test_ada}).to_csv("ada_test.csv", index=False)